# 00 · Data Preparation
### SCU Analytics Showdown — Feature Engineering Pipeline

## Context

**Good Nature Agro (GNA)** is a Zambian social enterprise that supports ~22,600 smallholder
farmers through a structured input-loan and crop buyback model.

This notebook ingests four raw operational datasets and engineers a single model-ready
`master_features.csv` that every subsequent notebook depends on.

**Four raw datasets:**
| Dataset | Description |
|---------|-------------|
| `farmer_details.csv` | Farmer demographics, location, tenure |
| `loan_details.csv` | Input package composition per farmer |
| `planting_survey.csv` | Planting dates, spacing, seed quantities |
| `buyback_details.csv` | Crop sold back to GNA (the target variable) |

In [1]:
import warnings, os
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Load raw datasets
DATA = "../Datasets"
farmers  = pd.read_csv(f"{DATA}/farmer_details.csv")
loans    = pd.read_csv(f"{DATA}/loan_details.csv")
planting = pd.read_csv(f"{DATA}/planting_survey.csv")
buyback  = pd.read_csv(f"{DATA}/buyback_details.csv")
# Strip whitespace from all column names
farmers.columns  = farmers.columns.str.strip()
loans.columns    = loans.columns.str.strip()
planting.columns = planting.columns.str.strip()
buyback.columns  = buyback.columns.str.strip()

# farmer_details uses farmer_fea_id — rename to match all other tables
farmers = farmers.rename(columns={"farmer_fea_id": "farmer_id"})

print(f"Farmers:   {len(farmers):,} rows × {farmers.shape[1]} cols")
print(f"Loans:     {len(loans):,} rows × {loans.shape[1]} cols")
print(f"Planting:  {len(planting):,} rows × {planting.shape[1]} cols")
print(f"Buyback:   {len(buyback):,} rows × {buyback.shape[1]} cols")

Farmers:   22,597 rows × 11 cols
Loans:     39,167 rows × 18 cols
Planting:  19,198 rows × 8 cols
Buyback:   24,439 rows × 19 cols


## 1 · Target Variable: Total Weight Sold

`total_weight` (kg) is what each farmer sold back to GNA in the 2024/25 season.
Farmers not in the buyback table are **non-sellers** (took loans but sold nothing back).

In [2]:
# Fix numeric columns in buyback
for col in ["total_weight","grade_a_weight","grade_b_weight","grade_c_weight",
            "total_net_weight","net_owed_to_farmer",
            "grade_a_cash_value","grade_b_cash_value","grade_c_cash_value"]:
    buyback[col] = pd.to_numeric(buyback[col], errors="coerce")

# Aggregate to farmer level
bbk = buyback.groupby("farmer_id").agg(
    total_weight_kg   = ("total_weight",       "sum"),
    grade_a_weight    = ("grade_a_weight",      "sum"),
    grade_b_weight    = ("grade_b_weight",      "sum"),
    grade_c_weight    = ("grade_c_weight",      "sum"),
    total_net_weight  = ("total_net_weight",    "sum"),
    net_owed_to_farmer= ("net_owed_to_farmer",  "sum"),
).reset_index()

# Grade A percentage (of actual graded weight, not total_weight which can be NaN)
grade_total = bbk["grade_a_weight"] + bbk["grade_b_weight"] + bbk["grade_c_weight"]
bbk["grade_a_pct"] = (bbk["grade_a_weight"] / grade_total.replace(0, np.nan)).clip(0, 1)

# Target: log1p transform handles skew + zeros
bbk["log_total_weight"] = np.log1p(bbk["total_weight_kg"])

# Non-sellers: all farmers in loan system but NOT in buyback
all_farmer_ids = farmers["farmer_id"].unique()
seller_ids     = set(bbk["farmer_id"])
non_seller_ids = set(all_farmer_ids) - seller_ids

print(f"Total farmers:  {len(all_farmer_ids):,}")
print(f"Sellers:        {len(seller_ids):,}  ({len(seller_ids)/len(all_farmer_ids):.1%})")
print(f"Non-sellers:    {len(non_seller_ids):,}  ({len(non_seller_ids)/len(all_farmer_ids):.1%})")
print(f"Target (log_total_weight) — mean: {bbk['log_total_weight'].mean():.3f}, std: {bbk['log_total_weight'].std():.3f}")

Total farmers:  22,597
Sellers:        18,687  (82.7%)
Non-sellers:    5,254  (23.3%)
Target (log_total_weight) — mean: 5.510, std: 1.352


## 2 · Farmer Features

In [3]:
# Parse dates
farmers["farmer_created_at"] = pd.to_datetime(farmers["farmer_created_at"], errors="coerce")
farmers["dob"]               = pd.to_datetime(farmers["dob"], errors="coerce")
REF = pd.Timestamp("2025-01-01")

farmers["age"]           = ((REF - farmers["dob"]).dt.days / 365.25).clip(18, 80)
farmers["days_as_member"]= (REF - farmers["farmer_created_at"]).dt.days.clip(0, None)
farmers["is_female"]     = (farmers["gender"].str.strip().str.lower() == "female").astype(int)
farmers["is_organic"]    = farmers["association"].str.lower().str.contains("organic", na=False).astype(int)

# Agro-ecological zones ordinal (I=1 → III=3)
zone_map = {"I": 1, "IIa": 2, "IIb": 2.5, "III": 3}
farmers["zone_ordinal"] = farmers["agroecological_zone"].map(zone_map).fillna(2)

print(farmers[["age","days_as_member","is_female","is_organic","zone_ordinal"]].describe().round(2))

            age  days_as_member  is_female  is_organic  zone_ordinal
count  22569.00        22597.00   22597.00    22597.00      22597.00
mean      40.28          149.11       0.45        0.00          2.25
std       13.13          140.32       0.50        0.05          0.47
min       18.00            0.00       0.00        0.00          1.00
25%       29.74           71.00       0.00        0.00          2.00
50%       38.98           82.00       0.00        0.00          2.00
75%       48.74          176.00       1.00        0.00          3.00
max       80.00          467.00       1.00        1.00          3.00


## 3 · Loan Package Features

In [4]:
# Program flags
programs = ["Seed","Source","Organic","Partnership","Asset","Pre Harvest","Family"]
for prog in programs:
    col = "has_" + prog.lower().replace(" ","_") + "_program"
    loans[col] = loans["program_name"].str.contains(prog, case=False, na=False).astype(int)

# Input flags (boolean columns already in loan_details)
input_cols = ["fertilizer","fungicide","gypsum","inoculant","insecticide","lime","seed_guard"]
for col in input_cols:
    loans[f"has_{col}"] = loans[col].astype(str).str.strip().str.lower().isin(["true","1","yes"]).astype(int)

# Aggregate to farmer
loan_agg = loans.groupby("farmer_id").agg(
    n_loan_packages      = ("id",              "count"),
    total_hectares       = ("package_hectares","sum"),
    total_inkind_repayment = ("package_in_kind_repayment","sum"),
    total_cash_repayment   = ("package_cash_repayment","sum"),
    total_down_payment   = ("initial_down_payment_value","sum"),
    n_crop_types_loaned  = ("crop_class_name", "nunique"),
    dominant_crop        = ("crop_class_name", lambda x: x.mode().iloc[0] if len(x) else "Unknown"),
    has_fertilizer       = ("has_fertilizer",  "max"),
    has_fungicide        = ("has_fungicide",   "max"),
    has_gypsum           = ("has_gypsum",      "max"),
    has_inoculant        = ("has_inoculant",   "max"),
    has_insecticide      = ("has_insecticide", "max"),
    has_lime             = ("has_lime",        "max"),
    has_seed_guard       = ("has_seed_guard",  "max"),
    has_source_program   = ("has_source_program",   "max"),
    has_seed_program     = ("has_seed_program",     "max"),
    has_organic_program  = ("has_organic_program",  "max"),
    has_partnership_program=("has_partnership_program","max"),
    has_asset_loan       = ("has_asset_program",    "max"),
    has_preharvest_loan  = ("has_pre_harvest_program","max"),
    has_family_package   = ("has_family_program","max"),
).reset_index()

# Input richness score (weighted: fertilizer/lime=3, fungicide/inoculant=2, others=1)
weights = {"has_fertilizer":3,"has_lime":3,"has_fungicide":2,"has_inoculant":2,
           "has_insecticide":1,"has_gypsum":1,"has_seed_guard":1}
loan_agg["input_richness_score"] = sum(loan_agg[c]*w for c,w in weights.items())
loan_agg["input_count"]          = sum(loan_agg[c] for c in weights)

print(f"Loan packages aggregated: {len(loan_agg):,} farmers")
print(f"Avg hectares per farmer:  {loan_agg['total_hectares'].median():.2f} ha")
print(f"Avg input richness score: {loan_agg['input_richness_score'].mean():.2f}")

Loan packages aggregated: 22,595 farmers
Avg hectares per farmer:  0.50 ha
Avg input richness score: 2.85


## 4 · Planting Survey Features

In [5]:
# Parse planting dates
planting["planting_date"] = pd.to_datetime(planting["planting_date"], errors="coerce")
SEASON_REF = pd.Timestamp("2024-11-01")   # start of 2024/25 season

planting["planting_doy"]         = (planting["planting_date"] - SEASON_REF).dt.days
planting["spacing_cm_btwn_rows"] = planting["spacing_cm_btwn_rows"].clip(10, 120)
planting["pct_multi_seed"]       = (planting["more_than_one_seed_in_hole"]
                                    .astype(str).str.lower().isin(["true","1","yes"])
                                    .astype(int))
planting["has_training"]         = (planting["rcvd_crop_training"]
                                    .astype(str).str.lower().isin(["true","1","yes"])
                                    .astype(int))

# Optimal spacing by crop (from agronomic guidelines)
OPTIMAL = {"Soy Bean":(45,50),"Groundnut":(30,45),"Sugar bean":(50,60),
           "Navy Bean":(50,60),"Cowpea":(60,75),"Sunflower":(60,75)}
DEFAULT = (45, 60)

def in_optimal(row):
    lo, hi = OPTIMAL.get(str(row.get("crop_planted","")), DEFAULT)
    s = row.get("spacing_cm_btwn_rows", np.nan)
    return 1 if (not pd.isna(s) and lo <= s <= hi) else 0

planting["spacing_optimal"] = planting.apply(in_optimal, axis=1)

survey_agg = planting.groupby("farmer_id").agg(
    qty_kgs_planted      = ("Qty_kgs_planted",      "sum"),
    avg_spacing          = ("spacing_cm_btwn_rows",  "mean"),
    pct_spacing_optimal  = ("spacing_optimal",        "mean"),
    has_training         = ("has_training",           "max"),
    pct_multi_seed       = ("pct_multi_seed",         "mean"),
    any_late_planting    = ("planting_doy",           lambda x: int((x > 90).any())),
    planting_doy         = ("planting_date",          lambda x: (x.min() - SEASON_REF).days if x.notna().any() else np.nan),
    planting_spread_days = ("planting_date",          lambda x: (x.max()-x.min()).days if x.notna().sum()>1 else 0),
    n_crops_planted      = ("crop_planted",           "nunique"),
).reset_index()

# days from loan to planting
account_dates = loans.groupby("farmer_id")["account_package_created_at"].min().reset_index()
account_dates["account_package_created_at"] = pd.to_datetime(
    account_dates["account_package_created_at"], errors="coerce")
account_dates["loan_doy"] = (account_dates["account_package_created_at"] - SEASON_REF).dt.days
survey_agg = survey_agg.merge(account_dates[["farmer_id","loan_doy"]], on="farmer_id", how="left")
survey_agg["days_loan_to_plant"] = survey_agg["planting_doy"] - survey_agg["loan_doy"]
survey_agg.loc[survey_agg["days_loan_to_plant"] < -180, "days_loan_to_plant"] = np.nan

# Seed density
survey_agg["seed_density_kg_per_ha"] = (survey_agg["qty_kgs_planted"] /
    survey_agg["qty_kgs_planted"].div(survey_agg["qty_kgs_planted"]).replace(0,np.nan)).clip(0,200)

print(f"Planting survey aggregated: {len(survey_agg):,} farmers")

Planting survey aggregated: 18,397 farmers


## 5 · Merge & Interaction Features

In [6]:
# Build full dataset: all farmers in loan system
base = farmers[["farmer_id","number_seasons","agroecological_zone","region_name",
                "age","days_as_member","is_female","is_organic","zone_ordinal"]].copy()

df = (base
      .merge(loan_agg,    on="farmer_id", how="left")
      .merge(survey_agg,  on="farmer_id", how="left")
      .merge(bbk[["farmer_id","total_weight_kg","log_total_weight",
                  "grade_a_pct","total_net_weight","net_owed_to_farmer"]],
             on="farmer_id", how="left"))

# Non-seller / has_buyback flags
df["non_seller"]  = (~df["farmer_id"].isin(seller_ids)).astype(int)
df["has_buyback"] = (df["farmer_id"].isin(seller_ids)).astype(int)
df["total_weight_kg"]  = df["total_weight_kg"].fillna(0)
df["log_total_weight"] = df["log_total_weight"].fillna(0)

# Yield per hectare
df["yield_per_ha"] = (df["total_weight_kg"] / df["total_hectares"].replace(0, np.nan)).clip(0, 15000)

# Interaction features
df["fertilizer_x_zone"]      = df["has_fertilizer"]      * df["zone_ordinal"]
df["lime_x_zone"]             = df["has_lime"]            * df["zone_ordinal"]
df["experience_x_training"]  = df["number_seasons"]       * df["has_training"].fillna(0)
df["rich_inputs_x_hectares"] = df["input_richness_score"] * df["total_hectares"]

print(f"Master dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Non-sellers:    {df['non_seller'].sum():,}  ({df['non_seller'].mean():.1%})")
print(f"Missing values: {df.isnull().sum().sum():,} across all columns")
df.head(3)

Master dataset: 22,597 rows × 56 columns
Non-sellers:    5,254  (23.3%)
Missing values: 89,919 across all columns


,farmer_id,number_seasons,agroecological_zone,region_name,age,days_as_member,is_female,is_organic,zone_ordinal,n_loan_packages,...,grade_a_pct,total_net_weight,net_owed_to_farmer,non_seller,has_buyback,yield_per_ha,fertilizer_x_zone,lime_x_zone,experience_x_training,rich_inputs_x_hectares
0,2504,3,IIa,Kasenengwa,32.205339,467,1,0,2.0,2.0,...,1.0,350.0,4750.0,0,1,550.0,0.0,0.0,3.0,4.0
1,2511,3,IIa,Kasenengwa,42.562628,466,0,0,2.0,1.0,...,1.0,1039.0,15585.0,0,1,2278.0,0.0,0.0,3.0,2.0
2,2512,2,IIa,Kasenengwa,28.386037,466,0,0,2.0,1.0,...,1.0,705.0,10575.0,0,1,1610.0,0.0,0.0,2.0,2.0


## 6 · Export

In [7]:
df.to_csv("../Datasets/master_features.csv", index=False)
print(f"Saved to.. /Datasets/master_features.csv  ({df.shape[0]:,} rows × {df.shape[1]} cols)")
print()
print("Key statistics:")
print(f"  Total farmers:          {len(df):,}")
print(f"  Sellers:                {df['has_buyback'].sum():,} ({df['has_buyback'].mean():.1%})")
print(f"  Non-sellers:            {df['non_seller'].sum():,} ({df['non_seller'].mean():.1%})")
print(f"  Median yield (sellers): {df[df['has_buyback']==1]['total_weight_kg'].median():.0f} kg")
print(f"  Median yield/ha:        {df[df['has_buyback']==1]['yield_per_ha'].median():.0f} kg/ha")

Saved to.. /Datasets/master_features.csv  (22,597 rows × 56 cols)

Key statistics:
  Total farmers:          22,597
  Sellers:                17,343 (76.7%)
  Non-sellers:            5,254 (23.3%)
  Median yield (sellers): 250 kg
  Median yield/ha:        404 kg/ha
